In [7]:
import pandas as pd
import optuna
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import os
from dotenv import load_dotenv
from supabase import create_client, Client

In [8]:
load_dotenv()
URL = os.getenv("SUPABASE_URL")
KEY = os.getenv("SUPABASE_KEY")
BUCKET_NAME = os.getenv("SUPABASE_BUCKET")
supabase: Client = create_client(URL, KEY)

In [9]:
cloud_path = f"{URL}/storage/v1/object/public/{BUCKET_NAME}/ulasan_ikd_preprocessed.csv"
print(f"Loading data from Supabase: {cloud_path}")

df = pd.read_csv(cloud_path)

# [KOREKSI FATAL]: Kembalikan NaN menjadi string kosong khusus untuk kolom ablasi
df = df.fillna("")

print("Data loaded successfully. Shape:", df.shape)

X = df['text_baseline']
y = df['label']

Loading data from Supabase: https://ffygtotdlnxhscukpmpu.supabase.co/storage/v1/object/public/dataset-ikd/ulasan_ikd_preprocessed.csv
Data loaded successfully. Shape: (34902, 7)


In [10]:
# 2. Data Splitting (Train: 64%, Val: 16%, Test: 20%)
# Train & Val digunakan untuk Optuna. Test disimpan murni untuk evaluasi akhir.
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp)

print("2. Vektorisasi TF-IDF...")
tfidf = TfidfVectorizer()
X_train_vec = tfidf.fit_transform(X_train)
X_val_vec = tfidf.transform(X_val)
X_test_vec = tfidf.transform(X_test)

2. Vektorisasi TF-IDF...


In [11]:
# optuna.delete_study(study_name="SVM_IKD_Tuning_v4", storage="sqlite:///svm_tuning.sqlite3")

In [12]:
# 3. Fungsi Objektif Optuna
def objective(trial):
    # ==========================================
    # 1. Ruang Pencarian Hyperparameter TF-IDF
    # ==========================================
    ngram_name = trial.suggest_categorical('ngram_range', ['unigram', 'bigram'])
    ngram_param = (1, 1) if ngram_name == 'unigram' else (1, 2)
    
    min_df_param = trial.suggest_int('min_df', 1, 5)
    max_df_param = trial.suggest_float('max_df', 0.7, 1.0)
    
    tfidf = TfidfVectorizer(
        ngram_range=ngram_param,
        min_df=min_df_param,
        max_df=max_df_param
    )
    
    X_train_vec = tfidf.fit_transform(X_train)
    X_val_vec = tfidf.transform(X_val)
    
    # ==========================================
    # 2. Ruang Pencarian Hyperparameter SVM
    # ==========================================
    c_param = trial.suggest_float('C', 0.1, 10.0, log=True)
    kernel_param = trial.suggest_categorical('kernel', ['linear', 'rbf'])
    
    if kernel_param == 'rbf':
        gamma_param = trial.suggest_categorical('gamma', ['scale', 'auto'])
        model = SVC(C=c_param, kernel=kernel_param, gamma=gamma_param, random_state=42)
    else:
        model = SVC(C=c_param, kernel=kernel_param, random_state=42)
        
    # ==========================================
    # 3. Training & Evaluasi
    # ==========================================
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_val_vec)
    macro_f1 = f1_score(y_val, y_pred, average='macro')
    
    return macro_f1

print("3. Memulai Hyperparameter Tuning dengan Optuna...")
db_file = "svm_tuning.sqlite3"
storage_url = f"sqlite:///{db_file}"
study_name = "SVM_IKD_Tuning_v4"

study = optuna.create_study(
    direction='maximize', 
    study_name=study_name, 
    storage=storage_url, 
    load_if_exists=True  
)

# Jalankan 20 trial
study.optimize(objective, n_trials=20)

print("\n=== HASIL TERBAIK OPTUNA ===")
print(f"Best Macro F1 (Validation): {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-11-03 19:30:14,076] A new study created in RDB with name: SVM_IKD_Tuning_v4


3. Memulai Hyperparameter Tuning dengan Optuna...


[I 2026-11-03 19:31:14,138] Trial 0 finished with value: 0.9244947432255003 and parameters: {'ngram_range': 'unigram', 'min_df': 5, 'max_df': 0.8534515861112026, 'C': 4.583228604434634, 'kernel': 'rbf', 'gamma': 'scale'}. Best is trial 0 with value: 0.9244947432255003.
[I 2026-11-03 19:31:47,793] Trial 1 finished with value: 0.9156780942620389 and parameters: {'ngram_range': 'bigram', 'min_df': 4, 'max_df': 0.7808003632511535, 'C': 4.110181801898402, 'kernel': 'linear'}. Best is trial 0 with value: 0.9244947432255003.
[I 2026-11-03 19:32:40,413] Trial 2 finished with value: 0.9138913466973067 and parameters: {'ngram_range': 'bigram', 'min_df': 2, 'max_df': 0.8156369403685628, 'C': 4.959430335405199, 'kernel': 'linear'}. Best is trial 0 with value: 0.9244947432255003.
[I 2026-11-03 19:33:05,368] Trial 3 finished with value: 0.9219960752611038 and parameters: {'ngram_range': 'bigram', 'min_df': 4, 'max_df': 0.7945504426699164, 'C': 0.8821969549465285, 'kernel': 'linear'}. Best is trial 0


=== HASIL TERBAIK OPTUNA ===
Best Macro F1 (Validation): 0.9262
Best Parameters:
  ngram_range: unigram
  min_df: 4
  max_df: 0.7960612188090677
  C: 2.0348244290341504
  kernel: rbf
  gamma: scale


```
=== HASIL TERBAIK OPTUNA - Trial 16 ===
Best Macro F1 (Validation): 0.9262
Best Parameters:
  ngram_range: unigram
  min_df: 4
  max_df: 0.7960612188090677
  C: 2.0348244290341504
  kernel: rbf
  gamma: scale
```

In [13]:
# 4. Evaluasi Final Menggunakan Parameter Terbaik di Data Test
print("\n4. Melatih ulang model terbaik pada seluruh data training & evaluasi di Data Test (Data Buta)...")
best_params = study.best_params

if best_params['kernel'] == 'rbf':
    best_svm = SVC(C=best_params['C'], kernel=best_params['kernel'], gamma=best_params['gamma'], random_state=42)
else:
    best_svm = SVC(C=best_params['C'], kernel=best_params['kernel'], random_state=42)

# Fit menggunakan gabungan Train + Val agar model mendapat data maksimal sebelum Test
X_train_val_vec = tfidf.fit_transform(X_temp)
X_test_final_vec = tfidf.transform(X_test)

best_svm.fit(X_train_val_vec, y_temp)
y_pred_final = best_svm.predict(X_test_final_vec)
final_macro_f1 = f1_score(y_test, y_pred_final, average='macro')

print("\n=== CLASSIFICATION REPORT (DATA TEST) ===")
print(classification_report(y_test, y_pred_final, digits=4))
print(f"Skor Uji Ablasi (Default) : 0.9269")
print(f"Skor Final Optuna (Tuned) : {final_macro_f1:.4f}")


4. Melatih ulang model terbaik pada seluruh data training & evaluasi di Data Test (Data Buta)...

=== CLASSIFICATION REPORT (DATA TEST) ===
              precision    recall  f1-score   support

           0     0.9176    0.9665    0.9414      4125
           1     0.9476    0.8746    0.9097      2856

    accuracy                         0.9290      6981
   macro avg     0.9326    0.9206    0.9256      6981
weighted avg     0.9299    0.9290    0.9284      6981

Skor Uji Ablasi (Default) : 0.9269
Skor Final Optuna (Tuned) : 0.9256


```
4. Melatih ulang model terbaik pada seluruh data training & evaluasi di Data Test (Data Buta)...

=== CLASSIFICATION REPORT (DATA TEST) ===
              precision    recall  f1-score   support

           0     0.9176    0.9665    0.9414      4125
           1     0.9476    0.8746    0.9097      2856

    accuracy                         0.9290      6981
   macro avg     0.9326    0.9206    0.9256      6981
weighted avg     0.9299    0.9290    0.9284      6981

Skor Uji Ablasi (Default) : 0.9269
Skor Final Optuna (Tuned) : 0.9256
```

In [14]:
print(f"Mengunggah riwayat eksperimen ({db_file}) ke Supabase...")
with open(db_file, 'rb') as f:
    supabase.storage.from_(BUCKET_NAME).upload(
        file=f, 
        path=db_file, 
        file_options={"x-upsert": "true"}
    )
print("✅ Riwayat Tuning SVM berhasil diamankan ke Supabase. Tim dapat mengaksesnya sekarang.")

Mengunggah riwayat eksperimen (svm_tuning.sqlite3) ke Supabase...
✅ Riwayat Tuning SVM berhasil diamankan ke Supabase. Tim dapat mengaksesnya sekarang.
